# 極座標の基底ベクトルと、クリストッフェル記号 $\Gamma$

[クリストッフェル記号とリーマン曲率テンソルのノート](../../../研究ノート/02_微分幾何/christoffel_riemann_intro.md#p6)（Part VI）の計算を、点を動かして確かめます。

**クリストッフェル記号の定義**は、基底ベクトルの変化率を、基底で展開した係数でした：

$$
\frac{\partial\mathbf e_i}{\partial q^j}=\Gamma^k{}_{ij}\,\mathbf e_k
$$

極座標 $(r,\theta)$ では、$\mathbf e_r=(\cos\theta,\sin\theta)$、$\mathbf e_\theta=r(-\sin\theta,\cos\theta)$ で、ゼロでない $\Gamma$ は
$\Gamma^r{}_{\theta\theta}=-r$、$\Gamma^\theta{}_{r\theta}=\Gamma^\theta{}_{\theta r}=1/r$ の3つだけです。

**使い方**：スライダーで点 $P$ を動かし、「動かす方向」を切り替えます。矢印の見方は、図の下に書いてあります。このノートブックは、ウィジェットを使うので、カーネルを動かして実行してください。

In [ ]:
import numpy as np
import sympy as sp
import plotly.graph_objects as go
import ipywidgets as W
from IPython.display import display

## 検算 1：$\Gamma$ を、基底ベクトルの微分から求める

SymPy で、定義 $\partial_j\mathbf e_i=\Gamma^k{}_{ij}\mathbf e_k$ から $\Gamma$ を計算します。

In [ ]:
r, th = sp.symbols("r theta", positive=True)
X = sp.Matrix([r * sp.cos(th), r * sp.sin(th)])
e = [X.diff(r), X.diff(th)]                  # e_r, e_θ（デカルト成分）
Jm = sp.Matrix.hstack(*e)                    # 列が基底ベクトル
coords, names = [r, th], ["r", "θ"]

Gamma = {}
for j, q in enumerate(coords):
    for i in range(2):
        coef = sp.simplify(Jm.inv() * e[i].diff(q))   # ∂_j e_i を (e_r, e_θ) で展開した係数 = Γ^k_{ij}
        for k in range(2):
            Gamma[(k, i, j)] = coef[k]

for (k, i, j), val in Gamma.items():
    if val != 0:
        print(f"Γ^{names[k]}_{{{names[i]}{names[j]}}} = {val}")

assert Gamma[(0, 1, 1)] == -r and Gamma[(1, 0, 1)] == 1 / r and Gamma[(1, 1, 0)] == 1 / r
assert sum(1 for v in Gamma.values() if v != 0) == 3

## 検算 2：数値でも確かめる

基底ベクトルを数値的に微分（中心差分）して、$\Gamma$ による展開と一致することを、いくつかの点で確かめます。

In [ ]:
def basis(r0, t0):
    return np.array([np.cos(t0), np.sin(t0)]), r0 * np.array([-np.sin(t0), np.cos(t0)])


def gamma_expansion(r0, t0, j):
    """∂_j e_i = Γ^k_{ij} e_k を、解析的な Γ で作る（i = r, θ の2つを返す）"""
    er, eth = basis(r0, t0)
    Gv = {(0, 1, 1): -r0, (1, 0, 1): 1 / r0, (1, 1, 0): 1 / r0}
    E = [er, eth]
    return [sum(Gv.get((k, i, j), 0.0) * E[k] for k in range(2)) for i in range(2)]


rng = np.random.default_rng(0)
h = 1e-6
for _ in range(5):
    r0, t0 = rng.uniform(0.5, 3), rng.uniform(0, 2 * np.pi)
    for j in range(2):
        dr, dt = (h, 0) if j == 0 else (0, h)
        num = [(a - b) / (2 * h) for a, b in zip(basis(r0 + dr, t0 + dt), basis(r0 - dr, t0 - dt))]
        exp = gamma_expansion(r0, t0, j)
        for i in range(2):
            assert np.allclose(num[i], exp[i], atol=1e-6), (r0, t0, i, j)
print("数値微分と Γ による展開が、すべての点で一致しました")

## 点を動かして見る

- **青**：$\mathbf e_r$（長さ 1）　**赤**：$\mathbf e_\theta$（長さ $r$）
- **緑**：$\mathbf e_r$ の変化率 $\partial_j\mathbf e_r$　**紫**：$\mathbf e_\theta$ の変化率 $\partial_j\mathbf e_\theta$（どちらも、対応する基底ベクトルの先端から描いています）
- 変化率の矢印は、基底の言葉に直すと、$\Gamma$ を係数とする組み合わせです。図の下に、その係数を表示します。

In [ ]:
MODES = ["θ 方向に動く（∂_θ）", "r 方向に動く（∂_r）"]


def arrow(p, q):
    return [p[0], q[0]], [p[1], q[1]]


def state(r0, t0, mode):
    """各トレースの座標と、説明文を返す"""
    P = np.array([r0 * np.cos(t0), r0 * np.sin(t0)])
    er, eth = basis(r0, t0)
    j = 1 if mode == MODES[0] else 0                       # 動かす方向の添字（0: r, 1: θ）
    d_er, d_eth = gamma_expansion(r0, t0, j)               # ∂_j e_r, ∂_j e_θ
    ang = np.linspace(0, 2 * np.pi, 200)
    coords = [
        (r0 * np.cos(ang), r0 * np.sin(ang)),                                 # r = 一定の円
        ([0, 3.4 * np.cos(t0)], [0, 3.4 * np.sin(t0)]),                       # θ = 一定の半直線
        arrow(P, P + er), arrow(P, P + eth),
        arrow(P + er, P + er + d_er), arrow(P + eth, P + eth + d_eth),
    ]
    if j == 1:
        text = (f"∂_θ e_r = Γ^θ_{{rθ}} e_θ = (1/r) e_θ = {1 / r0:+.3f} e_θ　　　"
                f"∂_θ e_θ = Γ^r_{{θθ}} e_r = −r e_r = {-r0:+.3f} e_r")
    else:
        text = (f"∂_r e_r = 0　　　"
                f"∂_r e_θ = Γ^θ_{{θr}} e_θ = (1/r) e_θ = {1 / r0:+.3f} e_θ")
    return coords, f"<b>r = {r0:.2f}, θ = {np.degrees(t0):.0f}°</b>　{text}"


def new_arrow_trace(color, name):
    return go.Scatter(x=[0, 0], y=[0, 0], mode="lines+markers", line=dict(color=color, width=4),
                      marker=dict(symbol="arrow", size=[0, 14], angleref="previous", color=color), name=name)


fig = go.FigureWidget(data=[
    go.Scatter(x=[], y=[], mode="lines", line=dict(color="#aaa", dash="dot"), name="r = 一定", hoverinfo="skip"),
    go.Scatter(x=[], y=[], mode="lines", line=dict(color="#aaa", dash="dash"), name="θ = 一定", hoverinfo="skip"),
    new_arrow_trace("#1f77b4", "e_r"),
    new_arrow_trace("#d62728", "e_θ（長さ r）"),
    new_arrow_trace("#2ca02c", "e_r の変化率"),
    new_arrow_trace("#9467bd", "e_θ の変化率"),
])
fig.add_trace(go.Scatter(x=[-3.6, 3.6], y=[0, 0], mode="lines", line=dict(color="#ddd", width=1), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=[0, 0], y=[-3.6, 3.6], mode="lines", line=dict(color="#ddd", width=1), showlegend=False, hoverinfo="skip"))
fig.update_layout(height=560, width=620, margin=dict(l=20, r=20, t=20, b=20),
                  xaxis=dict(range=[-3.6, 3.6], zeroline=False), yaxis=dict(range=[-3.6, 3.6], scaleanchor="x", zeroline=False),
                  legend=dict(orientation="h", y=-0.05))

r_slider = W.FloatSlider(value=1.5, min=0.4, max=2.5, step=0.05, description="r", continuous_update=True)
t_slider = W.FloatSlider(value=40, min=0, max=360, step=5, description="θ [度]", continuous_update=True)
mode_sel = W.ToggleButtons(options=MODES, description="動かす方向")
label = W.HTML()


def update(*_):
    coords, text = state(r_slider.value, np.radians(t_slider.value), mode_sel.value)
    with fig.batch_update():
        for tr, (x, y) in zip(fig.data[:6], coords):
            tr.x, tr.y = list(x), list(y)
    label.value = text


for w in (r_slider, t_slider, mode_sel):
    w.observe(update, names="value")
update()
display(W.VBox([mode_sel, r_slider, t_slider, fig, label]))

## 観察のポイント

- **θ 方向に動く**：緑の矢印（$\partial_\theta\mathbf e_r$）は、$\mathbf e_\theta$ と同じ向きです。点が円に沿って動くと、$\mathbf e_r$ の向きは、円の接線方向へ回ります。係数は $\Gamma^\theta{}_{r\theta}=1/r$ で、$r$ が小さいほど速く回ります。
- **θ 方向に動く**：紫の矢印（$\partial_\theta\mathbf e_\theta$）は、原点を向きます（$-r\,\mathbf e_r$）。これが $\Gamma^r{}_{\theta\theta}=-r$ で、ノートの Part VI §1 の計算です。
- **r 方向に動く**：$\mathbf e_r$ は変わらず（$\partial_r\mathbf e_r=0$）、$\mathbf e_\theta$ は向きを変えずに長くなります（$\partial_r\mathbf e_\theta=\mathbf e_\theta/r$）。
- 平面は平坦です。$\Gamma\ne0$ でも、曲率は 0 です（Part VI §3）。基底ベクトルが回って見えるのは、座標の取り方のせいです。